In [1]:
# !pip install tables

In [2]:
import pandas as pd
import glob
import os
from pathlib import Path
import h5py
from joblib import Parallel, delayed

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
replacement_map = {
    '├Ê': 'É',
    '├â': 'Ã',
    '├ü': 'Á',
    '├ç': 'Ç',
    '├ì': 'Í',
    '├Ò': 'é',
    '├│': 'ó',
    '├¡': 'í',
    '├ú': 'ã',
    '├í': 'á',
    '├º': 'ç',
    '├ô': 'Ó',
    '├õ': 'Ô',
    '├Ü': 'Ú',
    '├è': 'Ê',
    '├é': 'Â',
    '├┤': 'ô',
    '├║': 'ú',
    '├ò': 'õ',
    '├¬': 'ê',
    '├ó': 'â',
    '├╡': 'õ',
    '├Õ': 'Ú'
}

def fix_encoding(text, replacement_map):
    """
    Corrects encoding issues in a given text by replacing incorrectly encoded characters
    with their correct counterparts using a provided replacement map.

    Parameters:
        text (str): The input string that may contain incorrectly encoded characters.
        replacement_map (dict): A dictionary where keys are the incorrectly encoded
                                characters and values are the correct characters to
                                replace them with.

    Returns:
        str: The corrected text with all instances of incorrectly encoded characters
             replaced by their correct counterparts.

    Example:
        text = "This is a t├¬st with s├│me encoding probl├¡ms."
        corrected_text = fix_encoding(text, replacement_map)
        print(corrected_text)
        # Output: "This is a têst with sóme encoding problems."

    How It Works:
        1. Iterates over each key-value pair in the `replacement_map`.
        2. Replaces all occurrences of the incorrect character (key) in the input `text`
           with the correct character (value).
        3. Returns the corrected text after all replacements are made.

    Notes:
        - The function is case-sensitive, so ensure the keys in `replacement_map` match
          the exact incorrect characters in the text.
        - The `replacement_map` should be tailored to the specific encoding issues
          present in the text.
    """
    for incorrect, correct in replacement_map.items():
        text = text.replace(incorrect, correct)
    return text

In [5]:
# Function to read a single HDF5 file
def read_hdf_file(file_path, key_table):
    """
    Reads a single HDF5 file and returns the data.

    Parameters:
    file_path (str): The path to the HDF5 file.

    Returns:
    DataFrame: The data contained in the HDF5 file.
    """
    return pd.read_hdf(file_path, key=key_table)

In [6]:
def convert_state_abbreviations_to_names(df, abbreviation_column):
    # Dictionary mapping state abbreviations to their full names
    state_mapping = {
        'AC': 'ACRE',
        'AL': 'ALAGOAS',
        'AP': 'AMAPÁ',
        'AM': 'AMAZONAS',
        'BA': 'BAHIA',
        'CE': 'CEARÁ',
        'DF': 'DISTRITO FEDERAL',
        'ES': 'ESPÍRITO SANTO',
        'GO': 'GOIÁS',
        'MA': 'MARANHÃO',
        'MT': 'MATO GROSSO',
        'MS': 'MATO GROSSO DO SUL',
        'MG': 'MINAS GERAIS',
        'PA': 'PARÁ',
        'PB': 'PARAÍBA',
        'PR': 'PARANÁ',
        'PE': 'PERNAMBUCO',
        'PI': 'PIAUÍ',
        'RJ': 'RIO DE JANEIRO',
        'RN': 'RIO GRANDE DO NORTE',
        'RS': 'RIO GRANDE DO SUL',
        'RO': 'RONDÔNIA',
        'RR': 'RORAIMA',
        'SC': 'SANTA CATARINA',
        'SP': 'SÃO PAULO',
        'SE': 'SERGIPE',
        'TO': 'TOCANTINS'
    }
    reverse_state_mapping = {v: k for k, v in state_mapping.items()}

    # Create a new column with the full names of the states
    df[abbreviation_column] = df[abbreviation_column].str.upper().replace(state_mapping)

    # Create a new column with the full names of the states
    df['state_abbreviation'] = df[abbreviation_column].str.upper().replace(reverse_state_mapping)

    return df

# BRAZIL

In [8]:
brazil_folder_path = Path("./1 - Organized data gauge/")
# brazil_folder_path = "/content/drive/MyDrive/QualiBRain/Scripts e Dados/1 - Organized data gauge/"
brazil_h5_files = glob.glob(os.path.join(brazil_folder_path, "*.h5"))
print("quantity of files:", len(brazil_h5_files), "\nexample:")
for filename in brazil_h5_files:
    print("\\"+filename)

quantity of files: 4 
example:
\1 - Organized data gauge\CEMADEN_DAILY_2021_2024.h5
\1 - Organized data gauge\HIDROWEB_DAILY_1961_2020.h5
\1 - Organized data gauge\INMET_DAILY_2021_2024.h5
\1 - Organized data gauge\TELEMETRIA_DAILY_2021_2024.h5


In [9]:
table_key = 'table_info'

# Use parallel processing to read all files
brazil_list_info = Parallel(n_jobs=-1)(delayed(read_hdf_file)(filename, table_key) for filename in brazil_h5_files)

# Concatenate and remove duplicates
df_brazil_gauge_info = pd.concat(brazil_list_info).drop_duplicates(ignore_index=True)
del brazil_list_info

df_brazil_gauge_info['city'] = df_brazil_gauge_info['city'].apply(lambda x: x.upper())
df_brazil_gauge_info['state'] = df_brazil_gauge_info['state'].apply(lambda x: str(x).upper())
df_brazil_gauge_info = convert_state_abbreviations_to_names(df_brazil_gauge_info, 'state')
df_brazil_gauge_info = df_brazil_gauge_info.groupby('gauge_code').first().reset_index()
df_brazil_gauge_info


# df_brazil_gauge_info = df_brazil_gauge_info[['gauge_code',	'state',	'city',	'name_station',	'lat',	'long',	'responsible', 'source']]
df_brazil_gauge_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [10]:
table_key = 'table_data'

# Use parallel processing to read all files
brazil_list_data = Parallel(n_jobs=-1)(delayed(read_hdf_file)(filename, table_key) for filename in brazil_h5_files)

# Concatenate and remove duplicates
# df_brazil_gauge_data = pd.concat(brazil_list_data).drop_duplicates(ignore_index=True)
df_brazil_gauge_data = pd.concat(brazil_list_data)
del brazil_list_data

# df_brazil_gauge_data = df_brazil_gauge_data[['gauge_code',	'datetime',	'rain_mm']]
df_brazil_gauge_data['gauge_code'] = df_brazil_gauge_data['gauge_code'].astype(str)
df_brazil_gauge_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [11]:
df_brazil_gauge_data.dtypes

gauge_code            object
datetime      datetime64[ns]
rain_mm              float64
dtype: object

In [12]:
df_brazil_gauge_data['datetime'].max(), df_brazil_gauge_data['datetime'].min()

(Timestamp('2024-12-31 00:00:00'), Timestamp('1961-01-01 00:00:00'))

In [ ]:
df_brazil_gauge_data.to_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key='table_data', mode='w', complevel=9, complib='blosc', encoding='utf-8')
# df_brazil_gauge_data.to_hdf('BRAZIL_DAILY_1961_2024.h5', key='table_data', mode='w', complevel=9, complib='blosc', encoding='utf-8')
del df_brazil_gauge_data

In [10]:
df_brazil_gauge_info.to_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key='table_info', mode='r+', complevel=9, complib='blosc', encoding='utf-8')
# df_brazil_gauge_info.to_hdf('BRAZIL_DAILY_1961_2024.h5', key='table_info', mode='r+', complevel=9, complib='zlib', encoding='utf-8')
del df_brazil_gauge_info

In [ ]:
# df = pd.read_hdf('BRAZIL_DAILY_1961_2024.h5', key='table_info')
df_data = pd.read_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key='table_data')
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [16]:
# del df_data

In [11]:
df_info = pd.read_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key='table_info')
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


# data check

In [18]:
# df_data_qc = pd.read_hdf('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_QC.h5', key='table_data')
# df_data_qc

In [19]:
# from math import radians, sin, cos, sqrt, atan2

# df_teste = df_info[((df_info['lat'].astype(int) == -15) | (df_info['lat'].astype(int) == -16) |(df_info['lat'].astype(int) == -17) )
#                     & ((df_info['long'].astype(int) == -47) | (df_info['long'].astype(int) == -48) | (df_info['long'].astype(int) == -49) )
#                     & (df_info['source'] == 'HIDROWEB')]


# # Reference point
# ref_lat = -16.65
# ref_long = -48.65

# # Haversine function
# def haversine(lat1, lon1, lat2, lon2):
#     R = 6371  # Earth radius in km
#     lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
#     dlat = lat2 - lat1
#     dlon = lon2 - lon1
#     a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
#     c = 2 * atan2(sqrt(a), sqrt(1-a))
#     return R * c

# df_teste['distance'] = df_teste.apply(lambda row: haversine(ref_lat, ref_long, row['lat'], row['long']), axis=1)
# df_teste = df_teste.sort_values('distance')
# df_teste

In [20]:
# gauge_codes_nearest_10 = df_teste.nsmallest(10, 'distance')['gauge_code'].tolist()
# gauge_codes_nearest_10

In [21]:
# df_test = df_data_qc[df_data_qc['datetime'] == "2015-07-12"]
# df_test.sort_values(['rain_mm'], inplace=True)
# df_test = df_test[df_test['gauge_code'].isin(gauge_codes_nearest_10)]
# df_test

In [22]:
# df_teste_3 = pd.merge(df_teste, df_test, on='gauge_code', how='inner').sort_values(['distance'])
# df_teste_3

In [23]:
# df_station = df[
#     (df['gauge_code'] == '00651001')
#     & (df['datetime'] >= '2013-02-07')] #00651001 00655004
# df_station

In [24]:
# df_station['year'] = df_station['datetime'].dt.year
# df_station

In [25]:
# df_grouped = df_station.groupby(['gauge_code', 'year'])['rain_mm'].sum().reset_index()
# df_grouped

In [26]:
# df_station[['datetime', 'rain_mm']].plot(x='datetime', y='rain_mm')
